In [0]:
%run ../Includes/common_functions

In [0]:
%run ../Includes/config


In [0]:
v_data_source = dbutils.widgets.get("p_data_source")
v_file_date = dbutils.widgets.get("p_file_date")
print(v_data_source)
print(v_file_date)
print(raw_folder_path)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, concat, lit

from pyspark.sql.functions import concat,col


name_schema = StructType(fields=[StructField("forename", StringType(), True),
                                 StructField("surname", StringType(), True)
  
])

drivers_schema = StructType(fields=[StructField("driverId", StringType(), False),
                                    StructField("driverRef", StringType(), True),
                                    StructField("number", StringType(), True),
                                    StructField("code", StringType(), True),
                                    StructField("name", name_schema),
                                    StructField("dob", StringType(), True),
                                    StructField("nationality", StringType(), True),
                                    StructField("url", StringType(), True)  
])

drivers_df = (spark.read 
.schema(drivers_schema) 
.format("json")
.load(f"{raw_folder_path}/{v_file_date}/drivers.json")
)

In [0]:

drivers_with_columns_df = (drivers_df.withColumnRenamed("driverId", "driver_id") 
                                    .withColumnRenamed("driverRef", "driver_ref")  
                                    .withColumn("name", concat(col("name.forename"), lit(" "), col("name.surname")))
                                    .withColumn("ingestion_date", current_timestamp())
                                    .withColumn("data_source", lit(v_data_source)) 
                                    .withColumn("file_date", lit(v_file_date))
                                    
                               )

drivers_final_df = drivers_with_columns_df.drop(col("url"))
#display(drivers_final_df)
(drivers_final_df.write.mode("overwrite").option("mergeSchema","true").saveAsTable("f1.bronze.drivers"))
